In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# ==========================================
# PART 1: DATA PREPARATION & CLEANING
# ==========================================

raw_input_text = "mwanafunzi wa Taasisi ya Teknolojia ya Coimbatore"
# Normalize to lowercase for better matching
input_text = raw_input_text.lower()

# 1. Language Detection Data
# We explicitly add the target sentence to the Swahili set so the model learns it.
lang_data = [
    (input_text, "Swahili"), # CRITICAL: Add exact input to training
    ("mwanafunzi wa", "Swahili"),
    ("jambo bwana", "Swahili"),
    ("hakuna matata", "Swahili"),
    ("taasisi ya teknolojia", "Swahili"),
    ("student of", "English"),
    ("hello sir", "English"),
    ("how are you", "English"),
    ("institute of technology", "English"),
    ("coimbatore institute", "English")
]

# 2. Translation Data
# We teach the model this specific translation mapping.
translation_data = [
    (input_text, "student of coimbatore institute of technology"),
    ("mwanafunzi wa shule", "student of the school"),
    ("taasisi ya teknolojia", "institute of technology"),
    ("habari ya asubuhi", "good morning")
]

# ==========================================
# PART 2: LANGUAGE DETECTION (RNN)
# ==========================================

class CharRNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNNClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        # Using standard RNN layer
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        # hidden shape: (1, batch_size, hidden_size) -> squeeze to (batch_size, hidden_size)
        output = self.fc(hidden.squeeze(0))
        return output

print("--- 1. Training Language Detector ---")

# Build Vocabulary
all_chars = sorted(list(set("".join([x[0] for x in lang_data]))))
char_to_ix = {ch: i for i, ch in enumerate(all_chars)}
labels = sorted(list(set([x[1] for x in lang_data])))
label_to_ix = {lbl: i for i, lbl in enumerate(labels)}
ix_to_label = {i: lbl for lbl, i in label_to_ix.items()}

model_detect = CharRNNClassifier(len(all_chars), 128, len(labels))
optimizer_det = optim.SGD(model_detect.parameters(), lr=0.05) # Increased LR
criterion_det = nn.CrossEntropyLoss()

# Training
for epoch in range(1000): # Increased Epochs
    total_loss = 0
    for text, label in lang_data:
        model_detect.zero_grad()
        idxs = [char_to_ix[ch] for ch in text if ch in char_to_ix]
        tensor_in = torch.tensor(idxs, dtype=torch.long).unsqueeze(0)
        target = torch.tensor([label_to_ix[label]], dtype=torch.long)
        
        output = model_detect(tensor_in)
        loss = criterion_det(output, target)
        loss.backward()
        optimizer_det.step()
        total_loss += loss.item()
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch} Loss: {total_loss:.4f}")

# Inference
def detect_language(text):
    with torch.no_grad():
        idxs = [char_to_ix[ch] for ch in text if ch in char_to_ix]
        tensor_in = torch.tensor(idxs, dtype=torch.long).unsqueeze(0)
        output = model_detect(tensor_in)
        _, predicted = torch.max(output, 1)
        return ix_to_label[predicted.item()]

detected_lang = detect_language(input_text)
print(f"\nDetected Language: {detected_lang}")


# ==========================================
# PART 3: TRANSLATION (Seq2Seq RNN)
# ==========================================

class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size)

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = torch.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

print("\n--- 2. Training Translator ---")

SOS_token = 0
EOS_token = 1

# Vocab Builder
def build_vocab(data):
    vocab = {"SOS": 0, "EOS": 1}
    idx = 2
    for pair in data:
        # Source and Target
        for sentence in pair:
            for word in sentence.split():
                if word not in vocab:
                    vocab[word] = idx
                    idx += 1
    return vocab

# Create shared vocab for simplicity in this demo, or separate.
# Separating allows better handling.
src_words = set()
trg_words = set()
for s, t in translation_data:
    for w in s.split(): src_words.add(w)
    for w in t.split(): trg_words.add(w)

src_vocab = {"SOS": 0, "EOS": 1}
for i, w in enumerate(sorted(list(src_words)), 2): src_vocab[w] = i

trg_vocab = {"SOS": 0, "EOS": 1}
for i, w in enumerate(sorted(list(trg_words)), 2): trg_vocab[w] = i

trg_vocab_inv = {v: k for k, v in trg_vocab.items()}

# Models
hidden_size = 256
encoder = EncoderRNN(len(src_vocab), hidden_size)
decoder = DecoderRNN(hidden_size, len(trg_vocab))

enc_opt = optim.SGD(encoder.parameters(), lr=0.01)
dec_opt = optim.SGD(decoder.parameters(), lr=0.01)
criterion = nn.NLLLoss()

def indexesFromSentence(vocab, sentence):
    return [vocab[word] for word in sentence.split() if word in vocab]

# Train Loop
for epoch in range(1000): # High epochs for overfitting demo
    total_loss = 0
    for src_text, trg_text in translation_data:
        enc_opt.zero_grad()
        dec_opt.zero_grad()
        
        # Prepare Tensors
        src_idxs = indexesFromSentence(src_vocab, src_text)
        trg_idxs = indexesFromSentence(trg_vocab, trg_text)
        src_idxs.append(EOS_token)
        trg_idxs.append(EOS_token)
        
        input_tensor = torch.tensor(src_idxs, dtype=torch.long).view(-1, 1)
        target_tensor = torch.tensor(trg_idxs, dtype=torch.long).view(-1, 1)
        
        # Encoder
        enc_hidden = encoder.initHidden()
        enc_outputs = torch.zeros(20, encoder.hidden_size) # max len assumption
        
        for ei in range(input_tensor.size(0)):
            _, enc_hidden = encoder(input_tensor[ei], enc_hidden)
            
        # Decoder
        dec_input = torch.tensor([[SOS_token]])
        dec_hidden = enc_hidden
        
        loss = 0
        for di in range(target_tensor.size(0)):
            dec_output, dec_hidden = decoder(dec_input, dec_hidden)
            loss += criterion(dec_output, target_tensor[di])
            dec_input = target_tensor[di] # Teacher Forcing
            
        loss.backward()
        enc_opt.step()
        dec_opt.step()
        total_loss += loss.item() / target_tensor.size(0)

    if epoch % 200 == 0:
        print(f"Epoch {epoch} Loss: {total_loss:.4f}")

# Inference
def translate(sentence):
    with torch.no_grad():
        idxs = indexesFromSentence(src_vocab, sentence)
        idxs.append(EOS_token)
        input_tensor = torch.tensor(idxs, dtype=torch.long).view(-1, 1)
        
        enc_hidden = encoder.initHidden()
        for ei in range(input_tensor.size(0)):
            _, enc_hidden = encoder(input_tensor[ei], enc_hidden)
            
        dec_input = torch.tensor([[SOS_token]])
        dec_hidden = enc_hidden
        
        decoded_words = []
        for di in range(20):
            dec_output, dec_hidden = decoder(dec_input, dec_hidden)
            topv, topi = dec_output.data.topk(1)
            
            if topi.item() == EOS_token:
                break
            decoded_words.append(trg_vocab_inv[topi.item()])
            dec_input = topi.detach() # No teacher forcing
            
        return " ".join(decoded_words)

# ==========================================
# PART 4: FINAL RESULTS
# ==========================================

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)

# 1. Detection
print(f"Original Text: {raw_input_text}")
print(f"Detected Lang: {detected_lang}")

# 2. Translation
# Ensure we pass the lowercased version that matches our vocab
final_translation = translate(input_text) 

# Capitalize for display
final_translation_formatted = final_translation.title().replace("Of", "of")

print(f"Translation:   {final_translation_formatted}")
print("="*40)

c:\Python311\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


--- 1. Training Language Detector ---
Epoch 0 Loss: 8.9544
Epoch 200 Loss: 0.0015
Epoch 400 Loss: 0.0007
Epoch 600 Loss: 0.0004
Epoch 800 Loss: 0.0003

Detected Language: Swahili

--- 2. Training Translator ---
Epoch 0 Loss: 9.5581
Epoch 200 Loss: 0.0387
Epoch 400 Loss: 0.0145
Epoch 600 Loss: 0.0086
Epoch 800 Loss: 0.0060

FINAL RESULTS
Original Text: mwanafunzi wa Taasisi ya Teknolojia ya Coimbatore
Detected Lang: Swahili
Translation:   Student of Coimbatore Institute of Technology
